# Audited eICU sensitivity reanalysis v9
Fresh reproducible sensitivity rerun. Execute only on the authorized eICU host. This notebook is not a reconstruction of v7 and does not upload data. The default primary run uses TF-IDF; the CKKS roles are emulated in one process.


In [ ]:
# ============================================================================================
# PrivCal eICU reproducible sensitivity rerun v9; distinct from archived v7
#
# Evaluation of pooled calibration of clinical text predictors under cross-site drift
#
# Audit: enforce first-stay 24-hour landmark and correct zero-probability bins.
# Prespecified primary contrast is the site-level Brier score difference in
# target hospitals with 0–5 deaths, versus a regularized local intercept.
# Required files: ./eicu/patient.csv.gz ./eicu/diagnosis.csv.gz ./eicu/treatment.csv.gz
# IMPORTANT: Do NOT send eICU notes to hosted APIs. All model inference here is local.
# ============================================================================================

# --------------------------------------------------------------------------------------------
# 0. Environment: install required packages separately before running.
# This cell does not mutate the interpreter or install anything.
# --------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------
import os, re, json, time, math, hashlib, platform, sys
from pathlib import Path

import numpy as np
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_
if not hasattr(np, "object"):
    np.object = object
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "complex"):
    np.complex = complex

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.optimize import minimize_scalar, minimize

# --------------------------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------------------------
DEFAULT_EICU = "/home/spath004/Small Language Model Project/eicu"
RUN_VERSION = "real_eicu_v9_reproducibility_rerun"

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def input_fingerprint(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required restricted input is missing: {path}")
    return {"filename": path.name, "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)}

def required_input_path(folder, stem):
    folder = Path(folder)
    for suffix in (".csv.gz", ".csv"):
        candidate = folder / f"{stem}{suffix}"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Required restricted input is missing: {folder / stem}.csv[.gz]")

def installed_versions():
    from importlib.metadata import PackageNotFoundError, version
    result = {"python": sys.version}
    for package in ("numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "tenseal"):
        try:
            result[package] = version(package)
        except PackageNotFoundError:
            result[package] = "not-installed"
    return result

CFG = dict(
    seed=7,
    eicu_dir="./eicu" if Path("./eicu").exists() else DEFAULT_EICU,
    outdir=os.environ.get("PRIVCAL_OUTDIR", "./privcal_eicu_v9_sensitivities"),

    prediction_window_hours=24,
    min_site_n=120,
    require_real_ckks=True,
    source_frac=0.34,

    # --- backends: TF-IDF baseline + BioClinicalBERT under THREE pooling schemes (the ablation) ---
    model_backends=["tfidf_lr"],  # primary run; optionally enable BERT sensitivity
    require_real_clinical_lm=True,

    tfidf_max_features=30000,
    tfidf_min_df=2,

    # registry: backend id -> HF model + pooling scheme + its own embedding cache file
    clinical_lm_registry={
        "bioclinicalbert_cls_lr":       dict(model="emilyalsentzer/Bio_ClinicalBERT", pooling="cls"),
        "bioclinicalbert_mean_lr":      dict(model="emilyalsentzer/Bio_ClinicalBERT", pooling="mean"),
        "bioclinicalbert_last4mean_lr": dict(model="emilyalsentzer/Bio_ClinicalBERT", pooling="last4_mean"),
    },
    clinical_lm_max_length=256,
    clinical_lm_batch_size=32,
    clinical_lm_local_files_only=True,  # v9 never downloads models from a restricted-data run

    temp_grid=np.linspace(0.25, 5.0, 96),
    n_bins=12,
    multiseed_splits=[7, 11, 17, 23, 31],
    bootstrap_B=1000,
    stratum_bootstrap_B=2000,           # exploratory ECE interval; primary Brier separate

    local_calib_caps=[25, 50, 100, 200, "all"],
    # Sensitivity only; lambda=2.0 remains the archived v7 comparator definition.
    local_intercept_penalties=[0.5, 1.0, 2.0, 4.0, 8.0],
    min_local_iso_n=25,
    min_local_iso_pos=3,
    min_local_iso_neg=3,

    safety_alpha=0.05,
    safety_confidence=0.95,
    safety_min_low_coverage=0.01,
    safety_threshold_grid=np.linspace(0.01, 0.50, 250),
    safety_margin=0.00,

    max_heatmap_sites=8,
)

if os.environ.get("PYTHONHASHSEED") != "0":
    raise RuntimeError("Launch with PYTHONHASHSEED=0 for deterministic ordering")
run_started_utc = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
rng = np.random.default_rng(CFG["seed"])
np.random.seed(CFG["seed"])
outdir_path = Path(CFG["outdir"])
if outdir_path.exists():
    raise FileExistsError(f"Refusing to mix results in existing output directory: {outdir_path}")
input_fingerprints = {
    stem: input_fingerprint(required_input_path(CFG["eicu_dir"], stem))
    for stem in ("patient", "diagnosis", "treatment")
}
software_versions = installed_versions()
outdir_path.mkdir(parents=True)

# --------------------------------------------------------------------------------------------
# 3. Required CKKS arithmetic
# --------------------------------------------------------------------------------------------
try:
    import tenseal as ts
except ImportError as exc:
    raise ImportError("Install TenSEAL for the manuscript run; plaintext fallback is disabled") from exc
secret_ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192,
                        coeff_mod_bit_sizes=[60,40,40,60])
secret_ctx.global_scale = 2**40
public_ctx = ts.context_from(secret_ctx.serialize(save_secret_key=False))
assert not public_ctx.is_private()
HE_BACKEND = "TenSEAL CKKS; public evaluator context; key custodian emulated in one process"
REAL_CKKS = True

def he_secure_sum(vecs):
    assert len(vecs) >= 2, "Never reveal one site's aggregate"
    enc = [ts.ckks_vector(public_ctx, np.asarray(v,dtype=float).tolist()) for v in vecs]
    aggregate = enc[0]
    for ct in enc[1:]: aggregate += ct
    return np.asarray(aggregate.decrypt(secret_ctx.secret_key()))

# --------------------------------------------------------------------------------------------
# 4. Helper functions
# --------------------------------------------------------------------------------------------
def find_table(folder, stem):
    folder = Path(folder)
    for name in [f"{stem}.csv.gz", f"{stem}.csv"]:
        p = folder / name
        if p.exists():
            return p
    return None

def must_find(folder, stem):
    p = find_table(folder, stem)
    if p is None:
        raise FileNotFoundError(f"Missing {stem}.csv.gz or {stem}.csv in {Path(folder).resolve()}")
    return p

def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -40, 40)
    return 1 / (1 + np.exp(-z))

def clean_text(x):
    x = str(x).lower()
    x = re.sub(r"\[[^\]]+\]", " ", x)
    x = re.sub(r"[^a-z0-9.,;:/()%+\-\s]", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def ece(p, y, nb=None):
    if nb is None:
        nb = CFG["n_bins"]
    p, y = np.asarray(p), np.asarray(y)
    bins = np.linspace(0, 1, nb + 1)
    out = 0.0
    for i in range(nb):
        m = (p >= bins[i]) & (p <= bins[i + 1]) if i == 0 else (p > bins[i]) & (p <= bins[i + 1])
        if m.sum():
            out += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(out)

def adaptive_ece(p, y, nb=None):
    if nb is None:
        nb = CFG["n_bins"]
    p, y = np.asarray(p), np.asarray(y)
    o = np.argsort(p)
    p, y = p[o], y[o]
    out = 0.0
    for cp, cy in zip(np.array_split(p, nb), np.array_split(y, nb)):
        if len(cp):
            out += len(cp) / len(p) * abs(cp.mean() - cy.mean())
    return float(out)

def mce(p, y, nb=None):
    if nb is None:
        nb = CFG["n_bins"]
    p, y = np.asarray(p), np.asarray(y)
    bins = np.linspace(0, 1, nb + 1)
    gaps = []
    for i in range(nb):
        m = (p >= bins[i]) & (p <= bins[i + 1]) if i == 0 else (p > bins[i]) & (p <= bins[i + 1])
        if m.sum():
            gaps.append(abs(p[m].mean() - y[m].mean()))
    return float(max(gaps)) if gaps else 0.0

def brier(p, y):
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))

def nll(p, y):
    p = np.clip(np.asarray(p), 1e-7, 1 - 1e-7)
    y = np.asarray(y)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def auroc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan

def auprc(y, p):
    return float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else np.nan

def applyT(z, T):
    return sigmoid(z / T)

def fit_T(z, y):
    return float(
        minimize_scalar(
            lambda T: nll(applyT(z, T), y),
            bounds=(0.05, 10),
            method="bounded",
        ).x
    )

def wilson_upper_bound(k, n, confidence=0.95):
    if n <= 0:
        return 1.0
    if confidence >= 0.95:
        z = 1.96
    elif confidence >= 0.90:
        z = 1.645
    else:
        z = 1.28
    phat = k / n
    denom = 1 + z * z / n
    center = (phat + z * z / (2 * n)) / denom
    radius = z * math.sqrt((phat * (1 - phat) / n) + (z * z / (4 * n * n))) / denom
    return float(min(1.0, center + radius))

def safe_mean(x):
    x = pd.Series(x).dropna()
    return float(x.mean()) if len(x) else np.nan

def stable_hash_int(s):
    return int(hashlib.md5(str(s).encode("utf-8")).hexdigest()[:8], 16)

# --------------------------------------------------------------------------------------------
# 5. Load eICU
# --------------------------------------------------------------------------------------------
def load_real_eicu(cfg):
    d = Path(cfg["eicu_dir"])
    print("[loader] eICU dir:", d.resolve())

    patient_path = must_find(d, "patient")
    diagnosis_path = must_find(d, "diagnosis")
    treatment_path = must_find(d, "treatment")
    admissiondx_path = find_table(d, "admissionDx")

    pat = pd.read_csv(patient_path, low_memory=False)
    keep = [
        "patientunitstayid",
        "uniquepid",
        "hospitalid",
        "unitvisitnumber",
        "hospitaldischargestatus", "unitdischargeoffset",
    ]
    if set(keep)-set(pat.columns):
        raise ValueError(f"Required patient fields missing: {set(keep)-set(pat.columns)}")
    # Retain one deterministic observed ICU stay per unique patient before outcome
    # or landmark filtering. unitvisitnumber does not establish chronology across
    # distinct hospital admissions, so this is not called a patient's first ICU stay.
    pat=pat[keep].dropna(subset=["patientunitstayid","uniquepid","hospitalid"]).copy()
    for col in ["unitvisitnumber","unitdischargeoffset"]:
        pat[col]=pd.to_numeric(pat[col],errors="coerce")
    pat=pat.dropna(subset=["unitvisitnumber","unitdischargeoffset"])
    pat=(pat.sort_values(["uniquepid","unitvisitnumber","patientunitstayid"])
            .drop_duplicates("uniquepid"))
    pat=pat.dropna(subset=["hospitaldischargestatus","unitdischargeoffset"]).copy()
    landmark_min=int(cfg["prediction_window_hours"]*60)
    pat=pat[pat.unitdischargeoffset>=landmark_min].copy()
    print(f"[landmark] deterministic selected stays retained at 24h: {len(pat)}")

    pat["hospitaldischargestatus"] = pat["hospitaldischargestatus"].astype(str)
    pat["label"] = pat["hospitaldischargestatus"].str.lower().str.contains(
        "expired|death|dead"
    ).astype(int)
    pat["site"] = "hospital_" + pat["hospitalid"].astype(int).astype(str)

    window_min = None if cfg["prediction_window_hours"] is None else int(
        cfg["prediction_window_hours"] * 60
    )

    diag_cols = pd.read_csv(diagnosis_path, nrows=0).columns.tolist()
    diag_use = ["patientunitstayid", "diagnosisstring"]
    if "diagnosisoffset" in diag_cols:
        diag_use.append("diagnosisoffset")

    diag = pd.read_csv(
        diagnosis_path,
        usecols=lambda c: c in diag_use,
        low_memory=False,
    ).dropna(subset=["diagnosisstring"])

    if window_min is not None and "diagnosisoffset" in diag.columns:
        before = len(diag)
        diag = diag[
            (diag["diagnosisoffset"] >= 0)
            & (diag["diagnosisoffset"] <= window_min)
        ]
        print(f"[loader] diagnosis entries recorded by {cfg['prediction_window_hours']}h: {len(diag)}/{before}")
    elif window_min is not None:
        raise ValueError("diagnosisoffset missing, cannot enforce landmark")

    diag_g = (
        diag.groupby("patientunitstayid")["diagnosisstring"]
        .apply(lambda x: " ".join(map(str, x)))
        .reset_index(name="diagnosis_text")
    )

    tr_cols = pd.read_csv(treatment_path, nrows=0).columns.tolist()
    tr_use = ["patientunitstayid", "treatmentstring"]
    if "treatmentoffset" in tr_cols:
        tr_use.append("treatmentoffset")

    trt = pd.read_csv(
        treatment_path,
        usecols=lambda c: c in tr_use,
        low_memory=False,
    ).dropna(subset=["treatmentstring"])

    if window_min is not None and "treatmentoffset" in trt.columns:
        before = len(trt)
        trt = trt[
            (trt["treatmentoffset"] >= 0)
            & (trt["treatmentoffset"] <= window_min)
        ]
        print(f"[loader] treatment entries recorded by {cfg['prediction_window_hours']}h: {len(trt)}/{before}")
    elif window_min is not None:
        raise ValueError("treatmentoffset missing, cannot enforce landmark")

    trt_g = (
        trt.groupby("patientunitstayid")["treatmentstring"]
        .apply(lambda x: " ".join(map(str, x)))
        .reset_index(name="treatment_text")
    )

    data = pat.merge(diag_g, on="patientunitstayid", how="left")
    data = data.merge(trt_g, on="patientunitstayid", how="left")

    # admissionDx may be entered after 24h; omit this field in primary analysis.
    data["admissiondx_text"] = ""

    for c in ["diagnosis_text", "treatment_text", "admissiondx_text"]:
        data[c] = data[c].fillna("")
    # Filter genuine clinical content, not the fixed section-label string.
    actual_content = (data["diagnosis_text"].astype(str) + " " +
                      data["treatment_text"].astype(str)).map(clean_text)
    data = data[actual_content.str.len() > 20].copy()

    data["text"] = (
        "admission diagnosis: " + data["admissiondx_text"].astype(str)
        + " diagnosis: " + data["diagnosis_text"].astype(str)
        + " treatment: " + data["treatment_text"].astype(str)
    ).map(clean_text)

    data = data[data["text"].str.len() > 20].copy()
    site_counts = data["site"].value_counts()
    keep_sites = site_counts[site_counts >= cfg["min_site_n"]].index
    data = data[data["site"].isin(keep_sites)].copy()
    data = data.reset_index(drop=True)
    data["row_id"] = np.arange(len(data), dtype=int)

    print(
        f"[loader] rows={len(data)} hospitals={data['site'].nunique()} "
        f"mortality={data['label'].mean():.4f}"
    )

    return data[["row_id", "patientunitstayid", "site", "text", "label"]]

data = load_real_eicu(CFG)
SITES = sorted([str(s) for s in data["site"].unique()])

if len(SITES) < 4:
    raise ValueError(f"Need at least 4 hospitals after filtering. Found {len(SITES)}.")

site_table = (
    data.groupby("site")["label"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"sum": "deaths", "mean": "event_rate"})
    .sort_values("count", ascending=False)
)
site_table.to_csv(f"{CFG['outdir']}/site_summary.csv")

# --------------------------------------------------------------------------------------------
# 6. BioClinicalBERT local embedding backend (CLS / mean / last-4-layer mean)
# --------------------------------------------------------------------------------------------
def ensure_transformers_stack():
    os.environ["USE_TF"] = "0"
    os.environ["USE_FLAX"] = "0"
    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["TRANSFORMERS_NO_FLAX"] = "1"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    import numpy as np
    if not hasattr(np, "bool8"):
        np.bool8 = np.bool_
    if not hasattr(np, "object"):
        np.object = object
    if not hasattr(np, "int"):
        np.int = int
    if not hasattr(np, "float"):
        np.float = float
    if not hasattr(np, "complex"):
        np.complex = complex

    import torch
    import transformers
    return torch, transformers

def _pool(out, enc, pooling, torch):
    """Pool transformer outputs into one vector per example."""
    if pooling == "cls":
        return out.last_hidden_state[:, 0, :]
    if pooling == "mean":
        mask = enc["attention_mask"].unsqueeze(-1).float()
        return (out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
    if pooling in ("last4_mean", "last4_cls"):
        hs = out.hidden_states          # tuple length L+1, each [B,T,H]
        last4 = hs[-4:]
        if pooling == "last4_cls":
            return torch.cat([l[:, 0, :] for l in last4], dim=-1)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = [(l * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0) for l in last4]
        return torch.cat(pooled, dim=-1)
    raise ValueError(f"unknown pooling {pooling}")

def compute_hf_embeddings(texts, model_name, cache_file, max_length, batch_size, pooling, local_files_only):
    cache_path = Path(CFG["outdir"]) / cache_file
    meta_path = Path(str(cache_path) + ".meta.json")
    n = len(texts)
    digest = hashlib.sha256()
    for item in texts:
        digest.update(str(item).encode("utf-8")); digest.update(b"\0")
    corpus_sha256 = digest.hexdigest()

    if cache_path.exists() and meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        if (
            meta.get("n") == n
            and meta.get("corpus_sha256") == corpus_sha256
            and meta.get("model_name") == model_name
            and meta.get("max_length") == max_length
            and meta.get("pooling") == pooling
        ):
            print(f"[lm cache] loading cached embeddings ({pooling}): {cache_path}")
            return np.load(cache_path, mmap_mode="r")

    torch, transformers = ensure_transformers_stack()
    from transformers import AutoTokenizer, AutoModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    need_hidden = pooling.startswith("last4")
    print(f"[lm] loading {model_name} ({pooling}) on {device}")
    print(f"[lm] torch={torch.__version__} transformers={transformers.__version__} numpy={np.__version__}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=local_files_only, use_fast=True)
    model = AutoModel.from_pretrained(model_name, local_files_only=local_files_only)
    model.to(device)
    model.eval()

    sample = tokenizer(["test"], return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    sample = {k: v.to(device) for k, v in sample.items()}
    with torch.no_grad():
        out = model(**sample, output_hidden_states=need_hidden)
    hidden = int(_pool(out, sample, pooling, torch).shape[-1])

    tmp_path = str(cache_path) + ".tmp"
    emb = np.memmap(tmp_path, dtype="float32", mode="w+", shape=(n, hidden))
    t0 = time.time()

    for start in range(0, n, batch_size):
        batch_text = list(texts[start:start + batch_size])
        enc = tokenizer(batch_text, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc, output_hidden_states=need_hidden)
            pooled = _pool(out, enc, pooling, torch)
        emb[start:start + len(batch_text), :] = pooled.detach().cpu().numpy().astype("float32")

        if (start // batch_size) % 50 == 0:
            print(f"[lm:{pooling}] embedded {min(start + batch_size, n)}/{n} | {(time.time()-t0)/60:.1f} min")

    emb.flush()
    arr = np.asarray(emb)
    np.save(cache_path, arr)
    try:
        os.remove(tmp_path)
    except Exception:
        pass

    with open(meta_path, "w") as f:
        json.dump(dict(n=n,corpus_sha256=corpus_sha256,model_name=model_name,
                       max_length=max_length,pooling=pooling,
                       hidden=hidden,seconds=time.time() - t0),f,indent=2)
    print(f"[lm] saved embeddings ({pooling}): {cache_path}")
    return np.load(cache_path, mmap_mode="r")

# Compute (or load) embeddings for every BioClinicalBERT backend requested.
BACKEND_ARTIFACTS = {}
clinical_backends = [b for b in CFG["model_backends"] if b in CFG["clinical_lm_registry"]]
for bid in clinical_backends:
    spec = CFG["clinical_lm_registry"][bid]
    try:
        BACKEND_ARTIFACTS[bid] = compute_hf_embeddings(
            data["text"].values,
            model_name=spec["model"],
            cache_file=f"emb_{bid}.npy",
            max_length=CFG["clinical_lm_max_length"],
            batch_size=CFG["clinical_lm_batch_size"],
            pooling=spec["pooling"],
            local_files_only=CFG["clinical_lm_local_files_only"],
        )
    except Exception as e:
        print(f"[lm error] backend {bid} failed:", repr(e))
        if CFG["require_real_clinical_lm"]:
            raise SystemExit(f"{bid} failed and require_real_clinical_lm=True. Fix environment/model access first.")
        CFG["model_backends"] = [b for b in CFG["model_backends"] if b != bid]

# --------------------------------------------------------------------------------------------
# 7. Experiment functions
# --------------------------------------------------------------------------------------------
METHODS = [
    "Uncalibrated",
    "Global-T (source-only)",
    "Global-Platt (source-only)",
    "Global-BinnedIsotonic (source-only)",
    "PrivCal-A (secure-agg T)",
    "PrivCal-B (secure-agg isotonic)",
    "Per-site T (oracle)",
    "Per-site isotonic (oracle)",
]
SAFETY_METHODS = [
    "Uncalibrated",
    "Global-T (source-only)",
    "PrivCal-B (secure-agg isotonic)",
]

def train_backend_and_add_logits(backend, parts, SOURCE, split_seed):
    if backend == "tfidf_lr":
        tr = pd.concat([parts[s]["train"] for s in SOURCE], ignore_index=True)
        vec = TfidfVectorizer(
            max_features=CFG["tfidf_max_features"],
            min_df=CFG["tfidf_min_df"],
            ngram_range=(1, 2),
            sublinear_tf=True,
        ).fit(tr["text"])
        clf = LogisticRegression(max_iter=2500, class_weight="balanced", solver="liblinear",
                                 random_state=split_seed).fit(vec.transform(tr["text"]), tr["label"].values)
        for s in SITES:
            for part in ("calib", "test"):
                parts[s][f"logit_{part}"] = clf.decision_function(vec.transform(parts[s][part]["text"]))
                parts[s][f"y_{part}"] = parts[s][part]["label"].values
        return parts

    if backend in BACKEND_ARTIFACTS:                       # any BioClinicalBERT pooling backend
        emb = BACKEND_ARTIFACTS[backend]
        tr = pd.concat([parts[s]["train"] for s in SOURCE], ignore_index=True)
        X = np.asarray(emb[tr["row_id"].values], dtype="float32")
        clf = LogisticRegression(max_iter=2500, class_weight="balanced", solver="lbfgs",
                                 random_state=split_seed).fit(X, tr["label"].values)
        for s in SITES:
            for part in ("calib", "test"):
                idx = parts[s][part]["row_id"].values
                parts[s][f"logit_{part}"] = clf.decision_function(np.asarray(emb[idx], dtype="float32"))
                parts[s][f"y_{part}"] = parts[s][part]["label"].values
        return parts

    raise ValueError(f"Unknown backend: {backend}")

def fit_regularized_intercept(z, y, penalty):
    if penalty <= 0:
        raise ValueError("Intercept penalty must be positive")
    return float(minimize_scalar(
        lambda a: len(y) * nll(sigmoid(z + a), y) + penalty * a * a,
        bounds=(-8, 8), method="bounded").x)

def run_one_split(backend, split_seed, make_figures=False):
    split_rng = np.random.default_rng(split_seed)
    sh = np.array(SITES.copy())
    split_rng.shuffle(sh)

    n_src = max(2, int(CFG["source_frac"] * len(sh)))
    SOURCE = tuple(sorted(map(str, sh[:n_src])))  # stable order for training concatenation
    TARGET = [str(s) for s in SITES if str(s) not in SOURCE]
    if set(SOURCE).intersection(TARGET) or set(SOURCE).union(TARGET) != set(SITES):
        raise AssertionError("Source/target hospital partition audit failed")

    parts = {}
    for s in SITES:
        g = data[data.site == s].sample(frac=1, random_state=split_seed).reset_index(drop=True)
        n = len(g)
        n_train = int(0.50 * n)
        n_calib = int(0.25 * n)
        parts[s] = {
            "train": g.iloc[:n_train].copy(),
            "calib": g.iloc[n_train:n_train + n_calib].copy(),
            "test": g.iloc[n_train + n_calib:].copy(),
        }
        assigned = np.concatenate([parts[s]["train"]["row_id"].values,
                                  parts[s]["calib"]["row_id"].values,
                                  parts[s]["test"]["row_id"].values])
        if len(assigned) != len(np.unique(assigned)) or len(assigned) != len(g):
            raise AssertionError(f"Train/calibration/test overlap audit failed for {s}")

    parts = train_backend_and_add_logits(backend, parts, SOURCE, split_seed)

    zS = np.concatenate([parts[s]["logit_calib"] for s in SOURCE])
    yS = np.concatenate([parts[s]["y_calib"] for s in SOURCE])
    T_src = fit_T(zS, yS)
    # Sensitivity only: source labels fit both an intercept and positive slope.
    # This can repair the prevalence shift induced by class-weighted training.
    platt_opt = minimize(
        lambda v: nll(sigmoid(v[0] * zS + v[1]), yS),
        x0=np.array([1.0 / T_src, 0.0]), method="L-BFGS-B",
        bounds=[(1e-5, 20.0), (-20.0, 20.0)],
    )
    if not np.all(np.isfinite(platt_opt.x)) or not np.isfinite(platt_opt.fun):
        raise ArithmeticError("Source-only Platt fit failed")
    platt_slope, platt_intercept = map(float, platt_opt.x)

    curves = [
        np.array([nll(applyT(parts[s]["logit_calib"], T), parts[s]["y_calib"]) for T in CFG["temp_grid"]])
        * len(parts[s]["y_calib"])
        for s in SITES
    ]
    counts = [np.array([float(len(parts[s]["y_calib"]))]) for s in SITES]

    t0 = time.perf_counter()
    agg = he_secure_sum(curves)
    tot = he_secure_sum(counts)[0]
    he_t = time.perf_counter() - t0

    mean_curve = agg / tot
    plain_curve = np.sum(np.stack(curves), axis=0) / tot
    he_err = float(np.max(np.abs(mean_curve - plain_curve)))
    T_priv = float(CFG["temp_grid"][int(np.argmin(mean_curve))])
    T_pool = float(CFG["temp_grid"][int(np.argmin(plain_curve))])

    def site_bins(s):
        p = sigmoid(parts[s]["logit_calib"])
        y = parts[s]["y_calib"]
        bins = np.linspace(0, 1, CFG["n_bins"] + 1)
        n = np.zeros(CFG["n_bins"]); pos = np.zeros(CFG["n_bins"]); csum = np.zeros(CFG["n_bins"])
        for i in range(CFG["n_bins"]):
            m = (p >= bins[i]) & (p <= bins[i + 1]) if i == 0 else (p > bins[i]) & (p <= bins[i + 1])
            n[i] = m.sum()
            pos[i] = y[m].sum() if m.sum() else 0
            csum[i] = p[m].sum() if m.sum() else 0
        return n, pos, csum

    Nc = he_secure_sum([site_bins(s)[0] for s in SITES])
    Pc = he_secure_sum([site_bins(s)[1] for s in SITES])
    CS = he_secure_sum([site_bins(s)[2] for s in SITES])
    if max(np.max(abs(Nc-np.rint(Nc))),np.max(abs(Pc-np.rint(Pc))))>1e-3:
        raise ArithmeticError("CKKS integer count reconstruction failed")
    N,POS=np.rint(Nc).astype(int),np.rint(Pc).astype(int)
    assert np.all((N>=0)&(POS>=0)&(POS<=N))
    assert N.sum()==sum(len(parts[s]["y_calib"]) for s in SITES), "Bin coverage audit failed"
    assert POS.sum()==sum(int(parts[s]["y_calib"].sum()) for s in SITES), "Event count audit failed"
    v=N>0
    iso_he=IsotonicRegression(out_of_bounds="clip").fit(
        np.clip(CS[v]/N[v],0,1),POS[v]/N[v],sample_weight=N[v])

    # Matched-function-class control: a source-only binned isotonic map.
    src_bins = [site_bins(s) for s in SOURCE]
    N_src = np.sum([x[0] for x in src_bins], axis=0)
    POS_src = np.sum([x[1] for x in src_bins], axis=0)
    CS_src = np.sum([x[2] for x in src_bins], axis=0)
    v_src = N_src > 0
    if not np.any(v_src):
        raise ArithmeticError("Source-only binned isotonic has no occupied bins")
    iso_source = IsotonicRegression(out_of_bounds="clip").fit(
        np.clip(CS_src[v_src] / N_src[v_src], 0, 1),
        POS_src[v_src] / N_src[v_src], sample_weight=N_src[v_src])

    T_or = {s: fit_T(parts[s]["logit_calib"], parts[s]["y_calib"]) for s in SITES}
    iso_or = {
        s: IsotonicRegression(out_of_bounds="clip").fit(sigmoid(parts[s]["logit_calib"]), parts[s]["y_calib"])
        for s in SITES
    }

    def predict(m, s, split="test"):
        z = parts[s][f"logit_{split}"]
        p = sigmoid(z)
        if m == "Uncalibrated":
            return p
        if m == "Global-T (source-only)":
            return applyT(z, T_src)
        if m == "Global-Platt (source-only)":
            return sigmoid(platt_slope * z + platt_intercept)
        if m == "Global-BinnedIsotonic (source-only)":
            return iso_source.predict(p)
        if m == "PrivCal-A (secure-agg T)":
            return applyT(z, T_priv)
        if m == "PrivCal-B (secure-agg isotonic)":
            return iso_he.predict(p)
        if m == "Per-site T (oracle)":
            return applyT(z, T_or[s])
        if m == "Per-site isotonic (oracle)":
            return iso_or[s].predict(p)
        raise ValueError(m)

    rows = []
    for m in METHODS:
        for s in SITES:
            p = predict(m, s, "test")
            y = parts[s]["y_test"]
            rows.append(dict(
                backend=backend, split_seed=split_seed, method=m, site=s,
                site_type=("source" if s in SOURCE else "TARGET"),
                n_calib=len(parts[s]["calib"]), calib_deaths=int(parts[s]["y_calib"].sum()),
                n_test=len(parts[s]["test"]), test_deaths=int(parts[s]["y_test"].sum()),
                ECE=ece(p, y), aECE=adaptive_ece(p, y), MCE=mce(p, y), Brier=brier(p, y),
                NLL=nll(p, y), AUROC=auroc(y, p), AUPRC=auprc(y, p),
            ))
    per_site = pd.DataFrame(rows)
    summary = (
        per_site[per_site.site_type == "TARGET"]
        .groupby("method")[["ECE", "aECE", "MCE", "Brier", "NLL", "AUROC", "AUPRC"]]
        .mean().reindex(METHODS)
    )

    # ---- scarcity: secure cross-site vs purely-local calibration under capped local labels ----
    def fit_local_iso_or_fallback(s, cap):
        p_cal = sigmoid(parts[s]["logit_calib"]); y_cal = parts[s]["y_calib"]
        if cap != "all" and len(y_cal) > int(cap):
            local_rng = np.random.default_rng(split_seed + stable_hash_int((s, cap)))
            idx = local_rng.choice(len(y_cal), int(cap), replace=False)
            p_fit, y_fit = p_cal[idx], y_cal[idx]
        else:
            p_fit, y_fit = p_cal, y_cal
        n_fit = len(y_fit); pos = int(y_fit.sum()); neg = int(n_fit - pos)
        if n_fit >= CFG["min_local_iso_n"] and pos >= CFG["min_local_iso_pos"] and neg >= CFG["min_local_iso_neg"]:
            return "local_iso", IsotonicRegression(out_of_bounds="clip").fit(p_fit, y_fit), n_fit, pos
        if pos > 0 and neg > 0:
            logits_fit = np.log(np.clip(p_fit, 1e-7, 1 - 1e-7) / np.clip(1 - p_fit, 1e-7, 1))
            return "local_T", fit_T(logits_fit, y_fit), n_fit, pos
        return "local_none", None, n_fit, pos

    # Sensitivity only: fit the map using every other hospital's calibration
    # summary, excluding the target site's contribution. The subtraction here
    # is an internal analytic audit and is not a privacy-preserving deployment.
    loho_maps = {}
    for target_site in TARGET:
        n_s, d_s, sum_s = site_bins(target_site)
        other_n = N - n_s.astype(int)
        other_d = POS - d_s.astype(int)
        other_sum = CS - sum_s
        assert np.all(other_n >= 0) and np.all((other_d >= 0) & (other_d <= other_n))
        occupied = other_n > 0
        loho_maps[target_site] = IsotonicRegression(out_of_bounds="clip").fit(
            np.clip(other_sum[occupied] / other_n[occupied], 0, 1),
            other_d[occupied] / other_n[occupied], sample_weight=other_n[occupied],
        )

    scarcity_rows = []
    penalty_rows = []
    for cap in CFG["local_calib_caps"]:
        for s in TARGET:
            y_test = parts[s]["y_test"]
            p_secure = predict("PrivCal-B (secure-agg isotonic)", s, "test")
            p_pooled_temperature = predict("PrivCal-A (secure-agg T)", s, "test")
            p_source_platt = predict("Global-Platt (source-only)", s, "test")
            p_source_binned_iso = predict("Global-BinnedIsotonic (source-only)", s, "test")
            p_secure_loho = loho_maps[s].predict(sigmoid(parts[s]["logit_test"]))
            kind, model, n_fit, pos_fit = fit_local_iso_or_fallback(s, cap)
            base_p = sigmoid(parts[s]["logit_test"])
            if kind == "local_iso":
                p_local = model.predict(base_p)
            elif kind == "local_T":
                p_local = applyT(parts[s]["logit_test"], model)
            else:
                p_local = base_p
            zc=parts[s]["logit_calib"]/T_src; yc=parts[s]["y_calib"]
            if cap!="all" and len(yc)>int(cap):
                rr=np.random.default_rng(split_seed+stable_hash_int((s,cap)))
                ix=rr.choice(len(yc),int(cap),replace=False)
                zc,yc=zc[ix],yc[ix]
            offset = fit_regularized_intercept(zc, yc, penalty=2.0)
            p_intercept=sigmoid(parts[s]["logit_test"]/T_src+offset)
            # A stronger site-local intercept sensitivity starts from the
            # two-parameter source calibration. The primary baseline above
            # remains fixed to the v7 definition.
            offset_platt=float(minimize_scalar(
                lambda a:len(yc)*nll(sigmoid(
                    platt_slope*zc*T_src+platt_intercept+a),yc)+2*a*a,
                bounds=(-8,8),method="bounded").x)
            p_local_platt=sigmoid(platt_slope*parts[s]["logit_test"]
                                   +platt_intercept+offset_platt)
            scarcity_rows.append(dict(
                backend=backend, split_seed=split_seed, local_calib_cap=str(cap), site=s,
                n_calib_total=len(parts[s]["calib"]), calib_deaths_total=int(parts[s]["y_calib"].sum()),
                n_local_fit=n_fit, local_fit_deaths=pos_fit, local_fit_kind=kind,
                secure_agg_ECE=ece(p_secure,y_test), local_only_ECE=ece(p_local,y_test),
                secure_agg_Brier=brier(p_secure,y_test),local_only_Brier=brier(p_local,y_test),
                local_intercept_Brier=brier(p_intercept,y_test),
                pooled_temperature_Brier=brier(p_pooled_temperature,y_test),
                source_platt_Brier=brier(p_source_platt,y_test),
                source_binned_isotonic_Brier=brier(p_source_binned_iso,y_test),
                local_platt_intercept_Brier=brier(p_local_platt,y_test),
                pooled_excluding_site_Brier=brier(p_secure_loho,y_test),
                source_platt_NLL=nll(p_source_platt,y_test),
                local_platt_intercept_NLL=nll(p_local_platt,y_test),
                pooled_excluding_site_NLL=nll(p_secure_loho,y_test),
                secure_agg_aECE=adaptive_ece(p_secure,y_test),
                local_only_aECE=adaptive_ece(p_local,y_test),
                local_intercept_aECE=adaptive_ece(p_intercept,y_test),
                secure_better_than_intercept_Brier=brier(p_intercept,y_test)-brier(p_secure,y_test),
                secure_better_by=ece(p_local,y_test)-ece(p_secure,y_test),
                n_test=len(y_test), test_deaths=int(y_test.sum()),
            ))
            # Penalty sensitivity uses all available local calibration labels only.
            if cap == "all":
                for penalty in CFG["local_intercept_penalties"]:
                    penalized_offset = offset if penalty == 2.0 else fit_regularized_intercept(zc, yc, penalty)
                    p_penalized = sigmoid(parts[s]["logit_test"] / T_src + penalized_offset)
                    penalty_rows.append(dict(
                        backend=backend, split_seed=split_seed, site=s,
                        n_calib_total=len(parts[s]["calib"]),
                        calib_deaths_total=int(parts[s]["y_calib"].sum()),
                        n_test=len(y_test), test_deaths=int(y_test.sum()),
                        penalty=float(penalty),
                        pooled_isotonic_Brier=brier(p_secure, y_test),
                        local_intercept_Brier=brier(p_penalized, y_test),
                        local_minus_pooled_Brier=brier(p_penalized, y_test) - brier(p_secure, y_test),
                    ))
    scarcity = pd.DataFrame(scarcity_rows)
    penalty_sensitivity = pd.DataFrame(penalty_rows)

    def death_stratum(x):
        if x <= 5: return "00-05 local deaths"
        if x <= 15: return "06-15 local deaths"
        if x <= 30: return "16-30 local deaths"
        return "31+ local deaths"
    def size_stratum(x):
        if x < 250: return "<250 calib pts"
        if x < 500: return "250-499 calib pts"
        return "500+ calib pts"
    scarcity["death_stratum"] = scarcity["calib_deaths_total"].map(death_stratum)
    scarcity["size_stratum"] = scarcity["n_calib_total"].map(size_stratum)

    scarcity_by_death = (
        scarcity.groupby(["backend", "local_calib_cap", "death_stratum"])
        .agg(hospitals=("site", "nunique"), mean_calib_deaths=("calib_deaths_total", "mean"),
             secure_agg_ECE=("secure_agg_ECE", "mean"), local_only_ECE=("local_only_ECE", "mean"),
             secure_better_by=("secure_better_by", "mean"))
        .reset_index()
    )

    # ---- conservative Wilson-UCB per-hospital safety threshold ----
    def choose_conservative_tau(method, s):
        p_cal = predict(method, s, "calib"); y_cal = parts[s]["y_calib"]; best = None
        for tau in CFG["safety_threshold_grid"]:
            low = p_cal <= tau; n_low = int(low.sum())
            if n_low == 0:
                continue
            k_death = int(y_cal[low].sum()); cov = n_low / len(y_cal)
            ucb = wilson_upper_bound(k_death, n_low, CFG["safety_confidence"]); harm = k_death / n_low
            if cov >= CFG["safety_min_low_coverage"] and ucb <= (CFG["safety_alpha"] - CFG["safety_margin"]):
                best = (tau, cov, harm, ucb, n_low, k_death)
        return best

    safety_rows = []
    for m in SAFETY_METHODS:
        for s in TARGET:
            chosen = choose_conservative_tau(m, s)
            p_test = predict(m, s, "test"); y_test = parts[s]["y_test"]
            if chosen is None:
                safety_rows.append(dict(backend=backend, split_seed=split_seed, method=m, site=s, tau=np.nan,
                    test_low_coverage=0.0, test_conditional_harm=np.nan, test_missed_deaths=0,
                    test_low_count=0, test_total=len(y_test), unsafe=False, defer_all=True))
                continue
            tau = chosen[0]; low_test = p_test <= tau
            test_n_low = int(low_test.sum()); test_k = int(y_test[low_test].sum())
            test_harm = test_k / test_n_low if test_n_low else np.nan
            safety_rows.append(dict(backend=backend, split_seed=split_seed, method=m, site=s, tau=float(tau),
                test_low_coverage=float(test_n_low / len(y_test)),
                test_conditional_harm=float(test_harm) if not np.isnan(test_harm) else np.nan,
                test_missed_deaths=int(test_k), test_low_count=int(test_n_low), test_total=int(len(y_test)),
                unsafe=bool((not np.isnan(test_harm)) and test_harm > CFG["safety_alpha"]), defer_all=False))
    safety = pd.DataFrame(safety_rows)
    safety_summary = (
        safety.groupby(["backend", "method"])
        .agg(mean_test_low_coverage=("test_low_coverage", "mean"),
             mean_test_conditional_harm=("test_conditional_harm", safe_mean),
             pct_hospitals_unsafe=("unsafe", lambda x: 100 * np.mean(x)),
             hospitals_defer_all=("defer_all", "sum"),
             missed_deaths=("test_missed_deaths", "sum"))
        .reset_index()
    )

    # ---- matched-coverage safety control (identical for monotone methods) ----
    matched_rows = []
    for cov in [0.20, 0.35, 0.50]:
        for m in SAFETY_METHODS:
            P = np.concatenate([predict(m, s, "test") for s in TARGET])
            Y = np.concatenate([parts[s]["y_test"] for s in TARGET])
            k = max(1, int(cov * len(P))); idx = np.argsort(P)[:k]
            matched_rows.append(dict(backend=backend, split_seed=split_seed, method=m, coverage=cov,
                conditional_death_rate=float(Y[idx].mean()), missed_deaths=int(Y[idx].sum()),
                low_risk_count=int(k), total=int(len(Y))))
    matched_safety = pd.DataFrame(matched_rows)

    # ---- bootstrap CI (across target hospitals) for ECE per method ----
    boot_rng = np.random.default_rng(split_seed + 9999)
    target_sites = sorted(TARGET)
    boot_rows = []
    target_ece = per_site[per_site.site_type == "TARGET"][["method", "site", "ECE"]]
    for m in METHODS:
        vals = target_ece[target_ece.method == m].set_index("site").loc[target_sites]["ECE"].values
        means = [boot_rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(CFG["bootstrap_B"])]
        boot_rows.append(dict(backend=backend, split_seed=split_seed, method=m,
            ECE_mean=float(vals.mean()), CI95_low=float(np.percentile(means, 2.5)),
            CI95_high=float(np.percentile(means, 97.5))))
    boot = pd.DataFrame(boot_rows)

    if make_figures:
        make_primary_figures(backend, parts, TARGET, summary, plain_curve, mean_curve,
                             T_priv, T_src, T_pool, he_err, he_t, scarcity_by_death, safety_summary)

    return dict(backend=backend, split_seed=split_seed, SOURCE=SOURCE, TARGET=TARGET,
                summary=summary, per_site=per_site, boot=boot, scarcity=scarcity,
                scarcity_by_death=scarcity_by_death, safety=safety, safety_summary=safety_summary,
                matched_safety=matched_safety, penalty_sensitivity=penalty_sensitivity, T_src=T_src, T_priv=T_priv, T_pool=T_pool,
                he_t=he_t, he_err=he_err)

# --------------------------------------------------------------------------------------------
# 8. Figure generation
# --------------------------------------------------------------------------------------------
def make_primary_figures(backend, parts, TARGET, summary, plain_curve, mean_curve,
                         T_priv, T_src, T_pool, he_err, he_t, scarcity_by_death, safety_summary):
    safe_backend = backend.replace("/", "_")
    plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True, "grid.alpha": 0.25})

    fig, ax = plt.subplots(figsize=(9, 4))
    vals = [summary.loc[m, "ECE"] for m in METHODS]
    ax.bar(range(len(METHODS)), vals)
    ax.set_xticks(range(len(METHODS))); ax.set_xticklabels(METHODS, rotation=25, ha="right")
    ax.set_ylabel("mean ECE on target hospitals"); ax.set_title(f"Kill-test under cross-site drift: {backend}")
    for i, val in enumerate(vals):
        ax.text(i, val, f"{val:.3f}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{CFG['outdir']}/{safe_backend}_fig3_killtest.png"); plt.close()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].plot(CFG["temp_grid"], plain_curve, lw=3, alpha=0.5, label="plaintext pooled stats")
    axes[0].plot(CFG["temp_grid"], mean_curve, "--", lw=1.5, label="secure aggregate")
    axes[0].axvline(T_priv, color="blue", ls=":", label=f"T*={T_priv:.2f}")
    axes[0].set_xlabel("temperature"); axes[0].set_ylabel("aggregated NLL"); axes[0].legend(fontsize=7)
    axes[0].set_title("Secure aggregation equals plaintext aggregate")
    axes[1].axis("off")
    axes[1].text(0, 1, f"backend: {backend}\nHE backend:\n{HE_BACKEND}\n\n"
                 f"max |secure - plaintext| NLL: {he_err:.2e}\nsites aggregated: {len(SITES)}\n"
                 f"secure-agg time: {he_t*1000:.1f} ms\n\nT source-only: {T_src:.3f}\n"
                 f"T PrivCal-A: {T_priv:.3f}\nT plaintext-pool: {T_pool:.3f}",
                 va="top", family="monospace", fontsize=9)
    plt.tight_layout(); plt.savefig(f"{CFG['outdir']}/{safe_backend}_fig4_secure_aggregation.png"); plt.close()

    order = ["00-05 local deaths", "06-15 local deaths", "16-30 local deaths", "31+ local deaths"]
    caps = [str(c) for c in CFG["local_calib_caps"]]
    fig, axes = plt.subplots(1, len(caps), figsize=(3.2 * len(caps), 4), sharey=True)
    if len(caps) == 1:
        axes = [axes]
    for ax, cap in zip(axes, caps):
        sub = scarcity_by_death[scarcity_by_death.local_calib_cap == cap].set_index("death_stratum").reindex(order)
        ax.bar(range(len(order)), sub["secure_better_by"]); ax.axhline(0, color="black", lw=1)
        ax.set_title(f"local cap={cap}"); ax.set_xticks(range(len(order)))
        ax.set_xticklabels([o.split()[0] for o in order], rotation=45, ha="right")
        ax.set_ylabel("local ECE - secure ECE")
    fig.suptitle(f"Secure aggregation helps few-death hospitals? {backend}")
    plt.tight_layout(); plt.savefig(f"{CFG['outdir']}/{safe_backend}_fig5_death_stratified.png"); plt.close()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(safety_summary)), safety_summary["pct_hospitals_unsafe"])
    ax.set_xticks(range(len(safety_summary))); ax.set_xticklabels(safety_summary["method"], rotation=25, ha="right")
    ax.set_ylabel("% hospitals unsafe on test"); ax.set_title(f"Conservative Wilson-UCB safety: {backend}")
    plt.tight_layout(); plt.savefig(f"{CFG['outdir']}/{safe_backend}_fig6_conservative_safety.png"); plt.close()

# --------------------------------------------------------------------------------------------
# 9. Run all backends and splits
# --------------------------------------------------------------------------------------------
print(f"\n[data] mode=real rows={len(data)} sites={len(SITES)}")
print(site_table.head(20).round(4).to_string())
print(f"\n[backends] {CFG['model_backends']}")
print(f"[multiseed] running {len(CFG['multiseed_splits'])} source/target splits per backend ...")

all_results = []
for backend in CFG["model_backends"]:
    for i, split_seed in enumerate(CFG["multiseed_splits"]):
        print(f"\n[run] backend={backend} split_seed={split_seed}")
        all_results.append(run_one_split(backend, split_seed, make_figures=(i == 0)))

# Aggregate-only arithmetic verification for the added sensitivity run.
pd.DataFrame([dict(backend=r["backend"], split_seed=r["split_seed"],
                   pooled_loss_max_abs_error=r["he_err"],
                   encrypted_temperature=r["T_priv"],
                   plaintext_temperature=r["T_pool"],
                   aggregation_seconds=r["he_t"])
              for r in all_results]).to_csv(
    f"{CFG['outdir']}/he_arithmetic_audit.csv", index=False)

# --------------------------------------------------------------------------------------------
# 10. Save combined outputs
# --------------------------------------------------------------------------------------------
primary_results = {r["backend"]: r for r in all_results if r["split_seed"] == CFG["multiseed_splits"][0]}

per_site_all = pd.concat([r["per_site"] for r in all_results], ignore_index=True)
boot_all = pd.concat([r["boot"] for r in all_results], ignore_index=True)
scarcity_all = pd.concat([r["scarcity"] for r in all_results], ignore_index=True)
safety_all_site = pd.concat([r["safety"] for r in all_results], ignore_index=True)
matched_all = pd.concat([r["matched_safety"] for r in all_results], ignore_index=True)
penalty_all = pd.concat([r["penalty_sensitivity"] for r in all_results], ignore_index=True)

# Restricted per-site outputs remain on the authorized host at full precision.
per_site_all.to_csv(f"{CFG['outdir']}/all_per_site_metrics.csv", index=False)
boot_all.to_csv(f"{CFG['outdir']}/all_bootstrap_ci.csv", index=False)
scarcity_all.to_csv(f"{CFG['outdir']}/all_scarcity_site_level.csv", index=False)
safety_all_site.to_csv(f"{CFG['outdir']}/all_conservative_safety_site_level.csv", index=False)
matched_all.to_csv(f"{CFG['outdir']}/all_matched_coverage_safety.csv", index=False)
penalty_all.to_csv(f"{CFG['outdir']}/all_local_intercept_penalty_sensitivity.csv", index=False)

primary_summary_rows = []
for backend, r in primary_results.items():
    s = r["summary"].copy().reset_index()
    s.insert(0, "backend", backend)
    primary_summary_rows.append(s)
primary_summary = pd.concat(primary_summary_rows, ignore_index=True)
primary_summary.round(4).to_csv(f"{CFG['outdir']}/table1_primary_target_summary_by_backend.csv", index=False)

multi_ece = (per_site_all[per_site_all.site_type == "TARGET"]
             .groupby(["backend", "split_seed", "method"])["ECE"].mean().reset_index())
multi_ece_summary = (multi_ece.groupby(["backend", "method"])["ECE"].agg(["mean", "std"])
                     .rename(columns={"mean": "ECE_mean", "std": "ECE_sd"}).reset_index())
# discrimination (AUROC) summary across splits — this answers the pooling/AUROC question
auroc_summary = (per_site_all[per_site_all.site_type == "TARGET"]
                 .groupby(["backend", "split_seed", "method"])["AUROC"].mean().reset_index()
                 .groupby(["backend", "method"])["AUROC"].agg(["mean", "std"])
                 .rename(columns={"mean": "AUROC_mean", "std": "AUROC_sd"}).reset_index())
auroc_summary = auroc_summary[auroc_summary.method == "Uncalibrated"].drop(columns="method")
multi_ece.round(4).to_csv(f"{CFG['outdir']}/table1b_multiseed_ece_raw.csv", index=False)
multi_ece_summary.round(4).to_csv(f"{CFG['outdir']}/table1b_multiseed_ece_summary.csv", index=False)
auroc_summary.round(4).to_csv(f"{CFG['outdir']}/table1d_auroc_by_backend.csv", index=False)

scarcity_by_death_all = (scarcity_all.groupby(["backend", "local_calib_cap", "death_stratum"])
    .agg(hospitals_split_rows=("site", "count"), secure_agg_ECE=("secure_agg_ECE", "mean"),
         local_only_ECE=("local_only_ECE", "mean"), secure_better_by=("secure_better_by", "mean"))
    .reset_index())
scarcity_by_death_all.round(4).to_csv(f"{CFG['outdir']}/table3_multiseed_death_stratified.csv", index=False)

# Prespecified primary result: seed 7, TF-IDF, all local labels, 0–5 deaths.
primary_brier=scarcity_all[(scarcity_all.backend=="tfidf_lr") &
    (scarcity_all.split_seed==CFG["multiseed_splits"][0]) &
    (scarcity_all.local_calib_cap=="all") &
    (scarcity_all.calib_deaths_total<=5)].copy()
if len(primary_brier)==0:
    raise ValueError("No target hospitals in the primary 0–5 death stratum")
delta=primary_brier.secure_better_than_intercept_Brier.to_numpy(dtype=float)
rr=np.random.default_rng(971)
replicates=rr.choice(delta,size=(CFG["stratum_bootstrap_B"],len(delta)),
                     replace=True).mean(axis=1)
primary_brier_out=pd.DataFrame([dict(
    split_seed=CFG["multiseed_splits"][0],backend="tfidf_lr",
    stratum="0–5 calibration deaths",n_hospitals=len(delta),
    mean_pooled_Brier=primary_brier.secure_agg_Brier.mean(),
    mean_local_intercept_Brier=primary_brier.local_intercept_Brier.mean(),
    mean_local_minus_pooled_Brier=delta.mean(),
    CI95_low=np.percentile(replicates,2.5),
    CI95_high=np.percentile(replicates,97.5))])
primary_brier_out.to_csv(f"{CFG['outdir']}/table_primary_brier_few_events.csv",index=False)
primary_brier[["site","n_calib_total","calib_deaths_total","n_test",
               "secure_agg_Brier","local_intercept_Brier",
               "secure_better_than_intercept_Brier"]].to_csv(
                   f"{CFG['outdir']}/primary_brier_site_pairs.csv",index=False)

# ---- WITHIN-STRATUM BOOTSTRAP CI on the headline claim (primary seed, resample hospitals) ----
def within_stratum_bootstrap(primary_results, caps_to_report=("50", "all"), B=None, seed=0):
    B = B or CFG["stratum_bootstrap_B"]
    rg = np.random.default_rng(seed)
    order = ["00-05 local deaths", "06-15 local deaths", "16-30 local deaths", "31+ local deaths"]
    out = []
    for backend, r in primary_results.items():
        site_level = r["scarcity"]
        for cap in caps_to_report:
            sub = site_level[site_level.local_calib_cap == cap]
            for stratum in order:
                vals = sub[sub.death_stratum == stratum]["secure_better_by"].values
                if len(vals) == 0:
                    continue
                boots = [rg.choice(vals, size=len(vals), replace=True).mean() for _ in range(B)]
                lo, hi = np.percentile(boots, 2.5), np.percentile(boots, 97.5)
                out.append(dict(backend=backend, local_calib_cap=cap, death_stratum=stratum,
                    n_hospitals=int(len(vals)), mean_secure_better_by=round(float(vals.mean()), 4),
                    CI95_low=round(float(lo), 4), CI95_high=round(float(hi), 4),
                    CI_excludes_zero_in_favor_pooled=bool(lo > 0)))
    return pd.DataFrame(out)

stratum_ci = within_stratum_bootstrap(primary_results, caps_to_report=("50", "all"), seed=CFG["seed"])
stratum_ci.to_csv(f"{CFG['outdir']}/table5_within_stratum_bootstrap_ci.csv", index=False)

safety_summary_all = (safety_all_site.groupby(["backend", "split_seed", "method"])
    .agg(mean_test_low_coverage=("test_low_coverage", "mean"),
         mean_test_conditional_harm=("test_conditional_harm", safe_mean),
         pct_hospitals_unsafe=("unsafe", lambda x: 100 * np.mean(x)),
         hospitals_defer_all=("defer_all", "sum"), missed_deaths=("test_missed_deaths", "sum"))
    .reset_index())
safety_summary_multiseed = (safety_summary_all.groupby(["backend", "method"])
    .agg(mean_low_coverage=("mean_test_low_coverage", "mean"),
         mean_conditional_harm=("mean_test_conditional_harm", "mean"),
         pct_hospitals_unsafe=("pct_hospitals_unsafe", "mean"),
         hospitals_defer_all=("hospitals_defer_all", "mean"),
         missed_deaths=("missed_deaths", "mean")).reset_index())
safety_summary_multiseed.round(4).to_csv(f"{CFG['outdir']}/table4_multiseed_conservative_safety.csv", index=False)

# --------------------------------------------------------------------------------------------
# 11. Audit record and machine-readable manifest
# --------------------------------------------------------------------------------------------
audit_record = dict(
    primary_results_file="table_primary_brier_few_events.csv",
    primary_n_hospitals=int(primary_brier_out.n_hospitals.iloc[0]),
    primary_mean_local_minus_pooled_Brier=float(primary_brier_out.mean_local_minus_pooled_Brier.iloc[0]),
    note="Interpret after verifying the source data, encryption runtime, and hospital pairs."
)
with open(f"{CFG['outdir']}/run_manifest.json", "w") as f:
    json.dump(dict(project="Pooled calibration under outcome scarcity",
              version=RUN_VERSION, eicu_dir=str(Path(CFG["eicu_dir"]).resolve()),
              prediction_window_hours=CFG["prediction_window_hours"],
              rows=int(len(data)), sites=int(len(SITES)),
              splits=CFG["multiseed_splits"], model_backends=CFG["model_backends"],
              he_backend=HE_BACKEND, real_ckks=REAL_CKKS,
              clinical_lm_registry=CFG["clinical_lm_registry"],
              primary_result=audit_record,
              privacy_scope="Single-process arithmetic emulation, not a deployed multi-hospital protocol; no differential privacy claim",
              run_started_utc=run_started_utc,
              pythonhashseed=os.environ.get("PYTHONHASHSEED"),
              source_notebook_sha256=os.environ.get("PRIVCAL_NOTEBOOK_SHA256"),
              reviewed_v8_source_sha256="776b38c715e69248b4c52a16b0f8016e4ba8bde8faabc291d87b0d9177bd8327",
              input_fingerprints=input_fingerprints,
              software_versions=software_versions,
              source_site_order="sorted tuple"),f,indent=2)

# --------------------------------------------------------------------------------------------
# --------------------------------------------------------------------------------------------
# 13. Final report
# --------------------------------------------------------------------------------------------
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
print("\n" + "=" * 120)
print("PrivCal eICU v9 reproducible sensitivity rerun; primary Brier table controls interpretation")
print("=" * 120)
print(f"python {platform.python_version()} | HE backend: {HE_BACKEND} | real CKKS: {REAL_CKKS}")
print(f"rows={len(data)} hospitals={len(SITES)} splits={len(CFG['multiseed_splits'])} window={CFG['prediction_window_hours']}h")
print(f"model_backends={CFG['model_backends']}")

print("\n--- TABLE 1: primary seed TARGET summary by backend ---")
print(primary_summary.round(4).to_string(index=False))
print("\n--- TABLE 1b: ECE mean +/- SD across splits by backend ---")
print(multi_ece_summary.round(4).to_string(index=False))
print("\n--- TABLE 1d: AUROC (discrimination) by backend — answers the pooling question ---")
print(auroc_summary.round(4).to_string(index=False))
print("\n--- TABLE 3: MULTISEED death-stratified local-vs-secure ablation ---")
print(scarcity_by_death_all.round(4).to_string(index=False))
print("\n--- TABLE 5: Exploratory within-stratum ECE bootstrap intervals ---")
print(stratum_ci.to_string(index=False))
print("\n--- TABLE 4: MULTISEED conservative safety summary ---")
print(safety_summary_multiseed.round(4).to_string(index=False))
print("\n--- PRESPECIFIED PRIMARY BRIER COMPARISON: lower score is better ---")
print(primary_brier_out.to_string(index=False))

if not REAL_CKKS:
    print("\nWARNING: REAL_CKKS is False. Do not submit a privacy/HE paper until TenSEAL CKKS is active.")
clinical_ran = [b for b in CFG["model_backends"] if b in CFG["clinical_lm_registry"]]
if not clinical_ran:
    print("\nWARNING: No real clinical LM backend ran. Do not call this an SLM/clinical-LM paper.")

print("\noutputs ->", os.path.abspath(CFG["outdir"]))
print("=" * 120)